# ⚡ 06. Generation 3: Multivariate LSTM with Monte Carlo Dropout

Deep learning architecture with active dropout during test inference for empirical quantile sampling.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

class ProbabilisticLSTM(nn.Module):
    def __init__(self, n_features=9, hidden_dim=256, n_layers=2, dropout=0.3, horizon=24):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, horizon)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.dropout(out[:, -1, :])
        return self.fc(out)

def mc_dropout_predict(model, X_tensor, n_samples=200):
    model.train()  # Keep dropout stochastic
    with torch.no_grad():
        samples = torch.stack([model(X_tensor) for _ in range(n_samples)])
    q10 = torch.quantile(samples, 0.10, dim=0).cpu().numpy().flatten()
    q50 = torch.quantile(samples, 0.50, dim=0).cpu().numpy().flatten()
    q90 = torch.quantile(samples, 0.90, dim=0).cpu().numpy().flatten()
    return q10, q50, q90
